# ಪಾಠ 01 - AI ಏಜೆಂಟ್ಗಳ ಪರಿಚಯ

**ಆಧುನಿಕರಿಗೆ AI ಏಜೆಂಟ್ಗಳು** ಕೋನ್ಸುರ್‌ನ ಮೊದಲ ಪಾಠಕ್ಕೆ ಸ್ವಾಗತ!

**AI ಏಜೆಂಟ್** ಎಂದರೆ ದೊಡ್ಡ ಭಾಷಾ ಮಾದರಿಯನ್ನು (LLM) ಅದರ ಕೂರ್ಮಣಯಂತ್ರವಾಗಿ ಬಳಸುವ ಮತ್ತು ನೈಜ ವಿಶ್ವದಲ್ಲಿ *ಕ್ರಿಯೆಗಳು* ಕೈಗೊಳ್ಳುವ (APIಗಳ ಕರೆ, ಡೇಟಾಬೇಸ್ ಬೇಡಿಕೆ, ಅಥವಾ ಕೋಡ್ ಅವಹೇಳನ) ಕಾರ್ಯಕ್ರಮ, ಈ ಮೂಲಕ ಬಳಕೆದಾರರ ಪರವಾಗಿ ಗುರಿಯನ್ನು ಸಾಧಿಸುವದು.

ಈ ನೊಟ್ಬುಕ್‌ನಲ್ಲಿ ನೀವು ನಿಮ್ಮ ಮೊದಲ ಏಜೆಂಟ್ ಅನ್ನು ನಿರ್ಮಿಸುವಿರಿ: ರಜೆ ಸ್ಥಳಗಳನ್ನು ಶಿಫಾರಸು ಮಾಡುವ **ಪ್ರಯಾಣ ಏಜೆಂಟ್**. ಇದರ ಮೂಲಕ ನೀವು ಈ ಕೆಳಗಿನ ವಿಚಾರಗಳನ್ನು ಕಲಿಯುವಿರಿ:

1. **Microsoft Agent Framework** ಉಪಯೋಗಿಸಿ Azure AI Foundry Agent ಸೇವೆಗೆ ಸಂಪರ್ಕ ಸಾಧಿಸುವುದು.
2. ಏಜೆಂಟ್‌ಗೆ ಒಂದು **ಉಪಕರಣ** ಕೊಡುವುದು — ಕರೆಯಬಹುದಾದ ಸರಳ ಪೈಥಾನ್ ಕಾರ್ಯ.
3. ಏಜೆಂಟ್ ಅನ್ನು ಓಡಿಸುವುದು ಮತ್ತು ಅದರ ಪ್ರತಿಕ್ರಿಯೆಯನ್ನು ಪರಿಶೀಲಿಸುವುದು.
4. ಏಜೆಂಟ್‌ನ ಪ್ರತಿಕ್ರಿಯೆಯನ್ನು ಟೋಕನ್ ಪ್ರತಿ ಟೋಕನ್ ಆಗಿ ಸ್ಟ್ರೀಮ್ ಮಾಡುವುದು.


## ಸೆಟಪ್

ಈ ನೋಟ್ಬುಕ್ ಅನ್ನು ಚಾಲನೆ ಮಾಡುವ ಮೊದಲು, ನೀವು ಖಚಿತಪಡಿಸಿಕೊಳ್ಳಬೇಕು:

1. **ಡಿಪ್ಲಾಯ್ಡ್ ಆಗಿರುವ ಸಂಭಾಷಣೆ ಮಾದರಿಯೊಂದಿಗೆ (ಉದಾ. `gpt-4o-mini`) ಒಂದು Azure AI Foundry ಯೋಜನೆ**.
2. **Azure CLI ನೊಂದಿಗೆ ಲಾಗಿನ್ ಆಗಿರುವುದು** — ನಿಮ್ಮ ಟರ್ಮಿನಲ್‌ನಲ್ಲಿ `az login` ಅನ್ನು ಚಾಲನೆ ಮಾಡಿ.
3. **ಆವಶ್ಯಕ ಪರಿಸರ ಚರಗಳನ್ನು ಹೊಂದಿಸಲು:**
   - `AZURE_AI_PROJECT_ENDPOINT` — ನಿಮ್ಮ Azure AI Foundry ಯೋಜನೆಯ ಎಂಡ್‌ಪಾಯಿಂಟ್.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ನಿಮ್ಮ ಡಿಪ್ಲಾಯ್ಡ್ ಮಾದರಿಯ ಹೆಸರು.

ಕೆಳಗಿನ ಸೆಲ್ ನಿಮಗೆ ಬೇಕಾಗಿರುವ Python ಪ್ಯಾಕೇಜ್‌ಗಳನ್ನು ಸ್ಥಾಪಿಸುತ್ತದೆ.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ನಿಮ್ಮ ಮೊದಲ ಏಜೆಂಟ್ ಅನ್ನು ಸೃಷ್ಟಿಸುವುದು

ಏಜೆಂಟ್‌ಗೆ ಎರಡು ವಸ್ತುಗಳು ಬೇಕಾಗಿವೆ:

- **ಸೂಚನೆಗಳು** ಅವನಿಗೆ *ಅವನು ಯಾರು* ಮತ್ತು *ಹೇಗೆ چلಿಸಬೇಕು* ಎಂದು ತಿಳಿಸುವವು (ಸಿಸ್ಟಮ್ ಪ್ರಾಂಪ್ಟ್).
- **ಉಪಕರಣಗಳು** — Python ಕಾರ್ಯಗಳು `@tool` ಮೂಲಕ ಅಲಂಕರಿಸಲ್ಪಟ್ಟುವು, ಅವುಗಳನ್ನು ಏಜೆಂಟ್ ಮಾಹಿತಿ ಪಡೆಯಲು ಅಥವಾ ಕ್ರಿಯೆಗಳನ್ನು ನಿರ್ವಹಿಸಲು ಕರೆಸಬಹುದು.

ಕೆಳಗಿನ ಉದಾಹರಣೆಯಲ್ಲಿ ನಾವು ಜನಪ್ರಿಯ ಪ್ರವಾಸ ಗೋಲುಗಳ ಪಟ್ಟಿ ನೀಡುವ ಸರಳ ಉಪಕರಣವನ್ನುนิಡುವೆವು. ಬಳಕೆದಾರರು ಪ್ರವಾಸ ಸಲಹೆಗಳನ್ನು ಕೇಳಿದಾಗ ಏಜೆಂಟ್ ಈ ಉಪಕರಣವನ್ನು ಬಳಸುವುದು.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## ಸ್ಟ್ರೀಮಿಂಗ್ ಪ್ರತಿಕ್ರಿಯೆಗಳು

ಹೆಚ್ಚು ಸಂವಾದಾತ್ಮಕ ಅನುಭವಕ್ಕಾಗಿ ನೀವು ಏಜೆಂಟ್ ಪ್ರತಿಕ್ರಿಯೆಯನ್ನು **ಸ್ಟ್ರೀಮ್** ಮಾಡಬಹುದು. ಸಂಪೂರ್ಣ ಪ್ರತಿಕ್ರಿಯೆಗೆ ಕಾಯುವುದರ ಬದಲಾಗಿ, ಏಜೆಂಟ್ ತಯಾರಾದಂತೆ ಪಠ್ಯದ ತುಂಡುಗಳನ್ನು ನೀಡುತ್ತದೆ. ನೀವು ನಿಜ ಕಾಲದಲ್ಲಿ ಔಟ್‌ಪುಟ್ ಅನ್ನು ಪ್ರದರ್ಶಿಸಲು ಬಯಸುವ ಚಾಟ್ ಇಂಟರ್‌ಫೇಸ್‌ಗಳಲ್ಲಿ ಇದು ವಿಶೇಷವಾಗಿ ಉಪಯುಕ್ತವಾಗಿದೆ.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ ನೀವು ಹೇಗೆ ಎಂದು ಕಲಿತಿರಿ:

- **ಪ್ರೊವೈಡರ್ ರಚಿಸುವುದು** `AzureAIProjectAgentProvider` ಮೂಲಕ Azure AI Foundry Agent Service ಗೆ ಸಂಪರ್ಕಿಸಲು.
- **ಔزارವನ್ನು ವ್ಯಾಖ್ಯಾನಿಸುವುದು** `@tool` ಡೆಕೋರೇಟರ್ ನಿರ್ದಿಷ್ಟಪಡಿಸುವ ಮೂಲಕ ಆಗೆಂಟ್ ನಿಮ್ಮ Python ფუნქ್‌ಶನ್‌ಗಳನ್ನು ಕರೆ ಮಾಡಬಹುದು.
- **ಆಜೆಂಟ್ ಚಾಲನೆ ಮಾಡುವುದು** ಬಳಕೆದಾರ ಸಂದೇಶದೊಂದಿಗೆ ಮತ್ತು ಅದರ ಪ್ರತಿಕ್ರಿಯೆಯನ್ನು ಮುದ್ರಿಸುವುದು.
- **ಪ್ರತಿಕ್ರಿಯೆಗಳನ್ನು ಸ್ಟ್ರೀಮ್ ಮಾಡುವುದು** ನೈಜಕಾಲಾ ಔಟ್‌ಪುಟ್‌ಗಾಗಿ.

ಮುಂದಿನ ಪಾಠದಲ್ಲಿ ನಾವು ಏಜೆಂಟಿಕ್ ಫ್ರೇಮ್ವರ್ಕ್‌ಗಳನ್ನು ಹೆಚ್ಚಿನ ಆழದಲ್ಲಿ ಅನ್ವೇಷಿಸಿ ಮತ್ತು ಏಜೆಂಟ್‌ಗಳಿಗೆ ಹೆಚ್ಚು ಶಕ್ತಿಶಾಲಿ ಉಪಕರಣಗಳು ಮತ್ತು ಬಹು-ಹಂತ ತರ್ಕ ಸಾಮರ್ಥ್ಯಗಳನ್ನು ನೀಡುವುದು ಹೇಗೆ ಎಂದು ಕಲಿಯೋಣ.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಅನುಮೋದನೆ**:  
ಈ ದಾಖಲೆ AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ಪ್ರಾಮಾಣಿಕತೆಯತ್ತ ಪ್ರಯತ್ನಿಸಿದರೂ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ತಪ್ಪುಗಳು இருக்கಬಹುದು ಎಂಬುದನ್ನು ಗಮನದಲ್ಲಿ ಇಡಿಕೊಳ್ಳಿ. ಮೂಲ ಭಾಷೆಯ ದಾಖಲೆ ಅಧಿಕೃತ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗಿದೆ. ಈ ಅನುವಾದ ಬಳಕೆಯಿಂದ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪುಮತಗಳು ಅಥವಾ ಅರ್ಥಭ್ರಮೆಗಳಿಗಾಗಿ ನಾವು ಹೊಣೆಗಾರರಾಗುವುದಿಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
